<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment44_K_Anonymity_and_L_Diversity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# EXPERIMENT 6
# MEASURING RE-IDENTIFICATION RISK:
# K-ANONYMITY AND L-DIVERSITY
# ============================================================

import pandas as pd

print("=" * 75)
print("EXPERIMENT 6: K-ANONYMITY AND L-DIVERSITY")
print("=" * 75)


# ============================================================
# 1. BUILD SYNTHETIC DATASET
# ============================================================

def build_dataset():

    rows = [
        ("Anita", 29, "600001", "F", "Asthma"),
        ("Bala", 31, "600001", "M", "Diabetes"),
        ("Chitra", 34, "600002", "F", "Asthma"),
        ("Deepak", 37, "600002", "M", "Hypertension"),
        ("Esha", 42, "600003", "F", "Diabetes"),
        ("Farhan", 45, "600003", "M", "Asthma"),
        ("Gita", 47, "600003", "F", "Hypertension"),
        ("Hari", 52, "600004", "M", "Diabetes"),
        ("Iqbal", 55, "600004", "M", "Asthma"),
        ("Jaya", 58, "600004", "F", "Hypertension"),
        ("Kiran", 61, "600005", "M", "Diabetes"),
        ("Latha", 64, "600005", "F", "Asthma")
    ]

    return pd.DataFrame(
        rows,
        columns=[
            "name",
            "age",
            "pincode",
            "gender",
            "disease"
        ]
    )


# ============================================================
# 2. REMOVE DIRECT IDENTIFIERS
# ============================================================

def remove_direct_identifiers(
    df,
    identifiers=("name",)
):

    return df.drop(
        columns=[
            c for c in identifiers
            if c in df.columns
        ]
    )


# ============================================================
# 3. CALCULATE K-ANONYMITY
# ============================================================

def k_anonymity(
    df,
    quasi
):

    if df.empty:
        return 0

    group_sizes = (
        df.groupby(
            list(quasi)
        )
        .size()
    )

    return int(
        group_sizes.min()
    )


# ============================================================
# 4. FIND UNIQUE / RE-IDENTIFIABLE RECORDS
# ============================================================

def unique_rows(
    df,
    quasi
):

    if df.empty:
        return df

    group_sizes = (
        df.groupby(
            list(quasi)
        )
        .size()
    )

    unique_groups = group_sizes[
        group_sizes == 1
    ].index

    if len(quasi) == 1:

        return df[
            df[quasi[0]].isin(
                unique_groups
            )
        ]

    index = pd.MultiIndex.from_frame(
        df[list(quasi)]
    )

    return df[
        index.isin(unique_groups)
    ]


# ============================================================
# 5. GENERALISATION
# ============================================================

def generalise(
    df,
    age_bin=10,
    pin_keep=3
):

    out = df.copy()

    # Age bands
    lower = (
        out["age"] // age_bin
    ) * age_bin

    out["age"] = [
        f"{a}-{a + age_bin - 1}"
        for a in lower
    ]

    # Pincode generalisation
    out["pincode"] = (
        out["pincode"]
        .astype(str)
        .str[:pin_keep]
        +
        "*" * (6 - pin_keep)
    )

    return out


# ============================================================
# 6. SUPPRESSION
# ============================================================

def suppress_small_classes(
    df,
    quasi,
    k
):

    group_sizes = (
        df.groupby(
            list(quasi)
        )["disease"]
        .transform("size")
    )

    return df[
        group_sizes >= k
    ].copy()


# ============================================================
# 7. CALCULATE L-DIVERSITY
# ============================================================

def l_diversity(
    df,
    quasi,
    sensitive="disease"
):

    if df.empty:
        return 0

    diversity = (
        df.groupby(
            list(quasi)
        )[sensitive]
        .nunique()
    )

    return int(
        diversity.min()
    )


# ============================================================
# 8. CREATE DATASET
# ============================================================

df = build_dataset()

print("\nRAW DATASET")
print("-" * 75)

print(df.to_string(index=False))


# ============================================================
# 9. QUASI-IDENTIFIERS
# ============================================================

QI = (
    "age",
    "pincode",
    "gender"
)


# ============================================================
# 10. REMOVE NAME
# ============================================================

anon = remove_direct_identifiers(
    df
)

print("\n\nRELEASED TABLE WITHOUT NAME")
print("-" * 75)

print(
    anon.to_string(
        index=False
    )
)


# ============================================================
# 11. RAW K-ANONYMITY
# ============================================================

k_raw = k_anonymity(
    anon,
    QI
)

print(
    "\nReleased table WITHOUT "
    f"generalisation: k = {k_raw}"
)


# ============================================================
# 12. RE-IDENTIFICATION CHECK
# ============================================================

unique_records = unique_rows(
    anon,
    QI
)

print(
    "Re-identifiable records:",
    len(unique_records),
    "of",
    len(anon)
)


# ============================================================
# 13. 20-YEAR AGE GENERALISATION
# ============================================================

gen_20 = generalise(
    anon,
    age_bin=20,
    pin_keep=3
)

k_20 = k_anonymity(
    gen_20,
    QI
)

print(
    "\nAfter 20-year age bands "
    f"(still too weak): k = {k_20}"
)


# ============================================================
# 14. 40-YEAR AGE GENERALISATION
# ============================================================

gen_40 = generalise(
    anon,
    age_bin=40,
    pin_keep=3
)

k_40 = k_anonymity(
    gen_40,
    QI
)

print(
    "After 40-year age bands: "
    f"k = {k_40}"
)


# ============================================================
# 15. SUPPRESSION TO TARGET K = 2
# ============================================================

target_k = 2

sup = suppress_small_classes(
    gen_40,
    QI,
    target_k
)

k_final = k_anonymity(
    sup,
    QI
)

print(
    "After suppression "
    f"(k >= {target_k}): "
    f"k = {k_final} | "
    f"rows retained: {len(sup)} "
    f"of {len(gen_40)}"
)


# ============================================================
# 16. L-DIVERSITY
# ============================================================

l_final = l_diversity(
    sup,
    QI,
    "disease"
)

print(
    f"L-diversity: {l_final}"
)


# ============================================================
# 17. FINAL RELEASED TABLE
# ============================================================

print("\n\nFINAL RELEASED TABLE")
print("-" * 75)

print(
    sup.to_string(
        index=False
    )
)


# ============================================================
# 18. TEST CASES
# ============================================================

results = []


# TC1
results.append(
    (
        "TC1 raw table has k = 1",
        k_anonymity(anon, QI) == 1
    )
)


# TC2
results.append(
    (
        "TC2 every raw row re-identifiable",
        len(unique_records) == len(anon)
    )
)


# TC3
results.append(
    (
        "TC3 weak generalisation still k = 1",
        k_anonymity(gen_20, QI) == 1
    )
)


# TC3b
results.append(
    (
        "TC3b stronger generalisation raises k",
        k_anonymity(gen_40, QI)
        >
        k_anonymity(anon, QI)
    )
)


# TC4
results.append(
    (
        "TC4 suppression achieves k >= 2",
        k_anonymity(sup, QI) >= 2
    )
)


# TC5
results.append(
    (
        "TC5 suppression only removes rows",
        len(sup) <= len(gen_40)
    )
)


# TC6
results.append(
    (
        "TC6 name column removed",
        "name" not in anon.columns
    )
)


# TC7
results.append(
    (
        "TC7 l-diversity computed",
        l_diversity(sup, QI) >= 1
    )
)


# TC8
gen_coarse = generalise(
    anon,
    age_bin=40,
    pin_keep=2
)

k_coarse = k_anonymity(
    gen_coarse,
    QI
)

results.append(
    (
        "TC8 coarser bands give larger k",
        k_coarse >= k_40
    )
)


# ============================================================
# 19. DISPLAY TEST RESULTS
# ============================================================

print("\n\n")
print("=" * 75)
print("TEST RESULTS")
print("=" * 75)

for name, ok in results:

    print(
        f"{name:<48} -> "
        f"{'PASS' if ok else 'FAIL'}"
    )


passed = sum(
    ok for _, ok in results
)


print("-" * 75)

print(
    f"RESULT: {passed}/{len(results)} "
    "test cases passed"
)


# ============================================================
# 20. FINAL RESULT
# ============================================================

print("\n")
print("=" * 75)
print("EXPERIMENT 6 COMPLETED SUCCESSFULLY")
print("=" * 75)

print(
    f"Initial k-anonymity       : {k_raw}"
)

print(
    f"20-year generalisation   : {k_20}"
)

print(
    f"40-year generalisation   : {k_40}"
)

print(
    f"Final k-anonymity        : {k_final}"
)

print(
    f"Final l-diversity        : {l_final}"
)

print(
    f"Rows retained            : "
    f"{len(sup)}/{len(gen_40)}"
)

print(
    f"Test cases passed        : "
    f"{passed}/{len(results)}"
)

print("=" * 75)

EXPERIMENT 6: K-ANONYMITY AND L-DIVERSITY

RAW DATASET
---------------------------------------------------------------------------
  name  age pincode gender      disease
 Anita   29  600001      F       Asthma
  Bala   31  600001      M     Diabetes
Chitra   34  600002      F       Asthma
Deepak   37  600002      M Hypertension
  Esha   42  600003      F     Diabetes
Farhan   45  600003      M       Asthma
  Gita   47  600003      F Hypertension
  Hari   52  600004      M     Diabetes
 Iqbal   55  600004      M       Asthma
  Jaya   58  600004      F Hypertension
 Kiran   61  600005      M     Diabetes
 Latha   64  600005      F       Asthma


RELEASED TABLE WITHOUT NAME
---------------------------------------------------------------------------
 age pincode gender      disease
  29  600001      F       Asthma
  31  600001      M     Diabetes
  34  600002      F       Asthma
  37  600002      M Hypertension
  42  600003      F     Diabetes
  45  600003      M       Asthma
  47  600003